# Once Upon a Mirror, notebook 6: All the mirrors in one notebook

The whole game in one notebook: the estimate of a product, then Random, estimate then exploit, epsilon-greedy and
UCB, each in a few lines of Python, and at the end the race of every mirror on the game's race season at Cinderella's.
It is a revision of the five sections of the game. Notebooks 1 to 5 go deeper into each mirror, and you can do
this one on its own.

**How it works.**
1. Run the two **SETUP** cells first. They load the game's tools and the checks; you do not need to read them.
2. Then go down the notebook. Each task has a title starting with 🎯, an explanation, and two 💡 hints below it.
   Try first without the hints; read Hint 1 if you are stuck, and Hint 2 (the code with blanks to fill in) only after that.
3. In the 🎯 code cell, replace every `...` with your code: one to three lines per task. Run the cell.
4. Run the check cell below it: ✅ means your code works, ⏳ means it is not finished yet, and ❌ tells you what
   to fix.

Every other code cell is ready: just run it. It takes about 40 minutes.

In [1]:
# ======================================================================
# SETUP 1 OF 2: the game's tools (moods, tap, the board, estimate, pick_highest). Run it first.
# Just run this cell. You do not need to read or change it.
# ======================================================================
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

def estimate(row):
    """The estimate of a product: the average of its 1s and 0s (taps / showings); None when it was never shown."""
    if len(row) == 0:
        return None
    return sum(row) / len(row)


def estimates(rows):
    """The estimate of every product on the board, in the order of the rows."""
    return [estimate(row) for row in rows]

def pick_highest(values, next_number):
    """The row with the highest value. When several rows share the highest value, one of them at random, with a
    number from next_number: the rule of the game, so the same stream gives the same choice. A number is drawn only
    when there is a tie."""
    best = max(values)
    tied = [k for k in range(len(values)) if abs(values[k] - best) <= 1e-9]
    if len(tied) == 1:
        return tied[0]
    return tied[int(next_number() * len(tied))]


# For notebook 6: sqrt(n) is the square root of n, so sqrt(9) is 3.0
from math import sqrt

In [2]:
# ======================================================================
# SETUP 2 OF 2: the checks that tell you whether your code works. Run it second.
# Just run this cell. You do not need to read or change it.
# ======================================================================
# The checks of notebook 6, and play, which runs your mirror calmly.
# A check prints one short message and never stops the notebook.

# Each test is a season and what the game's mirror does on it, morning by morning: "shown" holds the row shown on
# each morning, "dots" her answer, "kinds" why the game's mirror chose that row, "notes" epsilon-greedy's spins,
# "draws" the numbers it took from next_number, and "slips" the seasons that the usual slips play.
_N6_TESTS = {'random_mirror': [{'about': "the race season at Cinderella's", 'call': "random_mirror(cinderella, season_moods(RACE), strategy_numbers(RACE, 'random'))", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 103993, 'extra': None, 'shown': '0323134003341103440130403401012024023122100133144043404042340020022313344230430010420143324301010020', 'dots': '0101000110000000001011010000101101001000000001000001000010110000000010101001011101101000100010100000', 'kinds': 'rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 100, 'slips': {'int0': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'k_minus': '0222023002231002330120302301011023022021100122133033303031230010021312233120320010310032323201010020'}}, {'about': "the race season at Rapunzel's", 'call': "random_mirror(rapunzel, season_moods(RACE), strategy_numbers(RACE, 'random'))", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 103993, 'extra': None, 'shown': '0323134003341103440130403401012024023122100133144043404042340020022313344230430010420143324301010020', 'dots': '0001000100001100000110000001111101001100000001100000000010010000000010101001001111101100000001110000', 'kinds': 'rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 100, 'slips': {'int0': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'k_minus': '0222023002231002330120302301011023022021100122133033303031230010021312233120320010310032323201010020'}}, {'about': "season 1 at Cinderella's", 'call': "random_mirror(cinderella, season_moods(1), strategy_numbers(1, 'random'))", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 1, 'extra': None, 'shown': '2232420232242330011120222443131034103122404410042212303023422103231214003301144324221204120032202344', 'dots': '0100000000001011110001100001000100001010011001110000011000000011010000100101010100010011001100011100', 'kinds': 'rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 100, 'slips': {'int0': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'k_minus': '2222320221131220011120221333121023102122303310031212202012311102131213002201133213211203110022202233'}}, {'about': "season 2 at Rapunzel's", 'call': "random_mirror(rapunzel, season_moods(2), strategy_numbers(2, 'random'))", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 2, 'extra': None, 'shown': '2211244123013104124021220344320111321423144330200313113400013314144110034321123110210241231034242030', 'dots': '0011000101010001100101111000000111110010100101000001100001000000101010000001100111011011001000010100', 'kinds': 'rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 100, 'slips': {'int0': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'k_minus': '1111133113012103023010210233220111211323133320100213102300002313033100033211013010210231121023231020'}}], 'estimate_then_exploit': [{'about': "the race season at Cinderella's", 'call': "estimate_then_exploit(cinderella, season_moods(RACE), strategy_numbers(RACE, 'estimate then exploit'), 5)", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 103993, 'extra': 5, 'shown': '0123401234012340123401234000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0001010000000101000010000001111101001110000011010011001011110000000111111001111111111100110011100000', 'kinds': 'ttttttttttttttttttttttttteeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 0, 'slips': {}}, {'about': "the race season at Rapunzel's", 'call': "estimate_then_exploit(rapunzel, season_moods(RACE), strategy_numbers(RACE, 'estimate then exploit'), 5)", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 103993, 'extra': 5, 'shown': '0123401234012340123401234111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0101001000010000100011000101111101101111001011110111011011111001010111111001111111111101111011110010', 'kinds': 'ttttttttttttttttttttttttteeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 0, 'slips': {}}, {'about': "season 16 at Cinderella's, with a tie after the test", 'call': "estimate_then_exploit(cinderella, season_moods(16), strategy_numbers(16, 'estimate then exploit'), 1)", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 16, 'extra': 1, 'shown': '0123433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'dots': '0101010010011000100101000000110100000001000011001111010110100111100000100100110010001100010011101100', 'kinds': 'ttttteeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 1, 'slips': {'every': '0123433133311133131333333333111111111111331133333333333333333333333333333333333333333333333333333333', 'index': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'p': '0123400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000'}}, {'about': "season 2 at Rapunzel's", 'call': "estimate_then_exploit(rapunzel, season_moods(2), strategy_numbers(2, 'estimate then exploit'), 2)", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 2, 'extra': 2, 'shown': '0123401234111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0100001001111011110101111011011111110010101111100101100111001100111011100111110111111011111010111111', 'kinds': 'tttttttttteeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 0, 'slips': {}}], 'epsilon_greedy': [{'about': "the race season at Cinderella's", 'call': "epsilon_greedy(cinderella, season_moods(RACE), strategy_numbers(RACE, 'epsilon-greedy'), 0.2)", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 103993, 'extra': 0.2, 'shown': '0123433433333333300000300000000000000000000010000140112000000010000000000000000000304030000040003000', 'dots': '0001000000001100111111011001111101001110000001010001000011110000000111111001111111110100110001100000', 'kinds': 'iiiiibbhbbbbbbbbbhbbbbhbbbbbbbbbbbbbbbbbbbbbhbbbbhhbhhhbbbbbbbhbbbbbbbbbbbbbbbbbbbhhhbhbbbbbhbbbhbbb', 'notes': [None, None, None, None, None, 0.32, 0.598, 0.073, 0.826, 0.569, 0.858, 0.229, 0.77, 0.222, 0.495, 0.889, 0.415, 0.082, 0.83, 0.535, 0.737, 0.571, 0.065, 0.29, 0.216, 0.924, 0.378, 0.729, 0.841, 0.221, 0.235, 0.282, 0.223, 0.901, 0.312, 0.751, 0.85, 0.201, 0.799, 0.963, 0.468, 0.969, 0.719, 0.759, 0.065, 0.28, 0.97, 0.996, 0.269, 0.042, 0.125, 0.482, 0.153, 0.167, 0.051, 0.37, 0.874, 0.575, 0.484, 0.846, 0.304, 0.856, 0.161, 0.203, 0.409, 0.307, 0.908, 0.819, 0.884, 0.732, 0.73, 0.277, 0.628, 0.309, 0.278, 0.39, 0.82, 0.336, 0.862, 0.243, 0.564, 0.484, 0.192, 0.092, 0.184, 0.533, 0.159, 0.469, 0.91, 0.268, 0.897, 0.653, 0.194, 0.412, 0.946, 0.247, 0.052, 0.996, 0.568, 0.976], 'draws': 111, 'slips': {'flip': '0123423443223322314311431443314002300124431143111112303323213321344244143334021224113041403130212312', 'reuse': '0123433000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'k_minus': '0123433333333333300000300000000000000000000010000130001000000010000000000000000000203020000030002000'}}, {'about': "the race season at Rapunzel's", 'call': "epsilon_greedy(rapunzel, season_moods(RACE), strategy_numbers(RACE, 'epsilon-greedy'), 0.2)", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 103993, 'extra': 0.2, 'shown': '0123434111111111011113111111111111111111111111111411121111111111111111111111111113041311111411131111', 'dots': '0101000111111101111110111101111101101111001011110011001011111001010111111001111110101001111011100010', 'kinds': 'iiiiibhbbbbbbbbbhbbbbhbbbbbbbbbbbbbbbbbbbbbhbbbbhhbhhhbbbbbbbhbbbbbbbbbbbbbbbbbbbhhhbhbbbbbhbbbhbbbb', 'notes': [None, None, None, None, None, 0.32, 0.073, 0.826, 0.569, 0.858, 0.229, 0.77, 0.222, 0.495, 0.889, 0.415, 0.082, 0.83, 0.535, 0.737, 0.571, 0.065, 0.29, 0.216, 0.924, 0.378, 0.729, 0.841, 0.221, 0.235, 0.282, 0.223, 0.901, 0.312, 0.751, 0.85, 0.201, 0.799, 0.963, 0.468, 0.969, 0.719, 0.759, 0.065, 0.28, 0.97, 0.996, 0.269, 0.042, 0.125, 0.482, 0.153, 0.167, 0.051, 0.37, 0.874, 0.575, 0.484, 0.846, 0.304, 0.856, 0.161, 0.203, 0.409, 0.307, 0.908, 0.819, 0.884, 0.732, 0.73, 0.277, 0.628, 0.309, 0.278, 0.39, 0.82, 0.336, 0.862, 0.243, 0.564, 0.484, 0.192, 0.092, 0.184, 0.533, 0.159, 0.469, 0.91, 0.268, 0.897, 0.653, 0.194, 0.412, 0.946, 0.247, 0.052, 0.996, 0.568, 0.976, 0.269], 'draws': 112, 'slips': {'flip': '0123423211401221143114314431140021001244111431111121011212131211442441431140212241131414031312123121', 'reuse': '0123430111111111100111101111111111111111111111011111010110101011111111011111111111111111111010001101', 'k_minus': '0123433111111111011113111111111111111111111111111310011111111111111111111111111112031211111311121111', 'index': '0123411411111113101111311111111111111111111111111141112111111111111111111111111111304131111141113111'}}, {'about': "season 1 at Cinderella's", 'call': "epsilon_greedy(cinderella, season_moods(1), strategy_numbers(1, 'epsilon-greedy'), 0.5)", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 1, 'extra': 0.5, 'shown': '0123410002110220330000103132001000040013130023333333331003404012133403004000004004112000001002004040', 'dots': '0100000010001001011011110010010101001011011001110010000000000011010000100101110100010011100100111100', 'kinds': 'iiiiibhhhhbbbhhbhhhbhhhbhhhhhhhbhbhhbbhhhhbbhbbbbbbbbhhbhhhbhbhhhhbhbbhbhbbbbbhbhhhhhbbbbbhbbhbbhbhb', 'notes': [None, None, None, None, None, 0.566, 0.36, 0.048, 0.007, 0.354, 0.871, 0.656, 0.759, 0.166, 0.365, 0.615, 0.123, 0.434, 0.339, 0.746, 0.403, 0.315, 0.061, 0.915, 0.365, 0.22, 0.294, 0.019, 0.083, 0.414, 0.172, 0.924, 0.116, 0.759, 0.16, 0.263, 0.996, 0.766, 0.129, 0.047, 0.149, 0.139, 0.832, 0.968, 0.37, 0.697, 0.643, 0.702, 0.717, 0.923, 0.537, 0.776, 0.533, 0.271, 0.208, 0.625, 0.404, 0.072, 0.118, 0.789, 0.359, 0.903, 0.147, 0.136, 0.333, 0.174, 0.756, 0.323, 0.96, 0.574, 0.039, 0.506, 0.202, 0.58, 0.866, 0.753, 0.88, 0.879, 0.033, 0.745, 0.064, 0.028, 0.15, 0.15, 0.094, 0.988, 0.974, 0.93, 0.602, 0.83, 0.011, 0.68, 0.781, 0.129, 0.624, 0.759, 0.167, 0.752, 0.424, 0.526], 'draws': 148, 'slips': {'flip': '0123411111114344134214244421130004440000040000004104343401020003040000000334001234030000000044400303', 'reuse': '0123411000001100100100002010020100101011100000200100000010000100010000120000000000101102000000100010', 'k_minus': '0123410002110220220000102122001000030013020010000000000210033030111203000030000030031120000000020030', 'index': '0123410002110002033030013313200100004301313002000000000031003404012130400004400004004112000001002004'}}, {'about': "season 2 at Rapunzel's", 'call': "epsilon_greedy(rapunzel, season_moods(2), strategy_numbers(2, 'epsilon-greedy'), 0.2)", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 2, 'extra': 0.2, 'shown': '0123401111111314111211111331111111111111111111113112111101121101114111011111111314111110111111111111', 'dots': '0100001111111011110101111001011111110010101111100101100101001100111011000111110010111010111010111111', 'kinds': 'iiiiihbbbbbbbhbhbbbhbbbbbhhbbbbbbbbbbbbbbbbbbbbbhbbhbbbbhbbhbbhbbbhbbbhbbbbbbbbhbhbbbbbhbbbbbbbbbbbb', 'notes': [None, None, None, None, None, 0.093, 0.632, 0.305, 0.324, 0.214, 0.512, 0.953, 0.826, 0.009, 0.209, 0.086, 0.839, 0.472, 0.741, 0.055, 0.655, 0.986, 0.213, 0.846, 0.23, 0.191, 0.157, 0.371, 0.509, 0.946, 0.765, 0.895, 0.913, 0.646, 0.812, 0.709, 0.217, 0.311, 0.517, 0.896, 0.952, 0.81, 0.645, 0.318, 0.907, 0.204, 0.394, 0.802, 0.161, 0.896, 0.776, 0.005, 0.53, 0.751, 0.362, 0.293, 0.053, 0.202, 0.648, 0.085, 0.937, 0.695, 0.183, 0.732, 0.551, 0.307, 0.07, 0.732, 0.839, 0.707, 0.023, 0.663, 0.631, 0.219, 0.415, 0.24, 0.236, 0.556, 0.995, 0.023, 0.491, 0.111, 0.58, 0.68, 0.978, 0.793, 0.844, 0.189, 0.784, 0.621, 0.588, 0.702, 0.706, 0.598, 0.594, 0.224, 0.74, 0.206, 0.972, 0.719], 'draws': 111, 'slips': {'flip': '0123411114011431311101443314411414021013140120331132141212441133211311000220212211221431221111304410', 'reuse': '0123400111111101101111011111101011111111111111111111110111011111001101110011101111001111111101101111', 'k_minus': '0123401111111213111111111321111111111111111111112112111101121101113111011111111213111110111111111111'}}], 'ucb': [{'about': "the race season at Cinderella's", 'call': "ucb(cinderella, season_moods(RACE), strategy_numbers(RACE, 'ucb'))", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 103993, 'extra': None, 'shown': '0123433000000001240000000003333330030000003000000030000000000004123000000000000000000000000000000000', 'dots': '0001000111011100001111011001101101001110000011010001001011110000000111111001111111111100110011100000', 'kinds': 'iiiiiuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 6, 'slips': {'nobonus': '0123433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'over_n': '0123431234034123000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'sqrt_n': '0123433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'index': '0123433000000000000000000012440000003340030000000000040033000030012000000000000000000000000000000000'}}, {'about': "the race season at Rapunzel's", 'call': "ucb(rapunzel, season_moods(RACE), strategy_numbers(RACE, 'ucb'))", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 103993, 'extra': None, 'shown': '0123413111111111131111111111111111111111111124401111111111111111143111111111111111111111111111111111', 'dots': '0101010111111101101111111101111101101111001001000111011011111001000111111001111111111101111011110010', 'kinds': 'iiiiiuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 5, 'slips': {'nobonus': '0123411311111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'over_n': '0123413111111112341011111111111111111111111111111111111111111111111111111111111111111111111111111111', 'sqrt_n': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'index': '0123413111111111131111111111111111111111110124411111111111111111134111111111111111111111111111111111'}}, {'about': "season 1 at Cinderella's", 'call': "ucb(cinderella, season_moods(1), strategy_numbers(1, 'ucb'))", 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'seed': 1, 'extra': None, 'shown': '0123411234000000000000000000000000001000000000000000000000000100000000000003333333333243333333000000', 'dots': '0100000000111111111011111011010101000011111011110010011000000011010100100101110100010001101100111110', 'kinds': 'iiiiiuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 5, 'slips': {'nobonus': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'over_n': '0123412130410000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'sqrt_n': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'index': '0123411023411110000000000000000000000000000000000000000000000233333334000000000000000000000000000000'}}, {'about': "season 2 at Rapunzel's", 'call': "ucb(rapunzel, season_moods(2), strategy_numbers(2, 'ucb'))", 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'seed': 2, 'extra': None, 'shown': '0123411111111111111111111111111111111400223111111111111201111111111111111111111111111111111111111111', 'dots': '0100011111111011110101111011011111110010100111100101100001001100111011100111110111111011111010111111', 'kinds': 'iiiiiuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu', 'notes': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'draws': 5, 'slips': {'nobonus': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'over_n': '0123411111111114102013443034111111111111114111111113301111111111111111111111111111111111111111111111', 'sqrt_n': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'index': '0123411111111111111111111111100111111233441111111111111034411111111111111411111111111111111111111111'}}]}
_N6_EST_TESTS = [[[0, 1, 0, 1, 1], 0.6], [[0, 0, 0, 0, 0], 0.0], [[0, 0, 0, 0, 0], 0.0], [[1, 0, 1, 0, 0], 0.4], [[0, 0, 0, 0, 0], 0.0], [[], None], [[1], 1.0], [[0, 0, 0], 0.0], [[0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], 0.4666666666666667], [[1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0], 0.47619047619047616]]
_N6_SUCCESS = {'my_estimate': "my_estimate works on 10 rows, as the game's estimate: [0, 1, 0, 1, 1] gives 0.6, and a row never shown gives None.", 'random_mirror': "random_mirror works on 4 seasons, morning by morning as in the game. On the race season at Cinderella's: 31 taps, a rate of 0.31, the game's number.", 'estimate_then_exploit': "estimate_then_exploit works on 4 seasons, morning by morning as in the game. On the race season at Cinderella's with n = 5: 46 taps, a rate of 0.46, the game's number.", 'epsilon_greedy': "epsilon_greedy works on 4 seasons, morning by morning as in the game. On the race season at Cinderella's with eps = 0.2: 47 taps, a rate of 0.47, the game's number.", 'ucb': "ucb works on 4 seasons, morning by morning as in the game. On the race season at Cinderella's: 52 taps, a rate of 0.52, the game's number."}
_N6_SLIPS = {'random_mirror': {'int0': 'every product is row 0. int(next_number()) is 0 for any number below 1, so multiply first, then take int: product = int(next_number() * len(p)).', 'k_minus': 'your mirror never shows the last row. Multiply by len(p), the number of products: int(next_number() * len(p)) gives every row from 0 to 4.'}, 'estimate_then_exploit': {'every': 'your mirror picks again on every remaining morning. Estimate then exploit picks the winner once, right after the test and before the loop: winner = pick_highest(estimates(rows), next_number).', 'index': "on a tie the game's mirror draws a number to choose among the tied rows. pick_highest(estimates(rows), next_number) does exactly that.", 'p': "your mirror picked the highest probability in p, which only the Queen's mirror may read. Estimate then exploit picks the highest estimate on its board: winner = pick_highest(estimates(rows), next_number)."}, 'epsilon_greedy': {'flip': 'your mirror took a card from the hat when the spin was above eps. The hat is the small slice: if spin < eps.', 'reuse': 'the card from the hat is a new number, drawn after the spin: product = int(next_number() * len(p)).', 'k_minus': 'your cards never reach the last row. Multiply by len(p): int(next_number() * len(p)).', 'index': "on a tie the game's mirror draws a number to choose among the tied rows: product = pick_highest(estimates(rows), next_number)."}, 'ucb': {'nobonus': "your mirror shows the highest estimate and forgets the bonus. UCB's score is estimate(row) + bonus, with bonus = 1 / sqrt(len(row)).", 'over_n': "your bonus is 1 / len(row). UCB's bonus is one divided by the square root of the showings: 1 / sqrt(len(row)).", 'sqrt_n': "your mirror favours the rows with the most dots. UCB's bonus is biggest for the rows with the fewest dots, and it is added: estimate(row) + 1 / sqrt(len(row)).", 'index': "on a tie the game's mirror draws a number to choose among the tied rows: product = pick_highest(scores, next_number)."}}
_N6_STREAMS = {'random_mirror': 'random', 'estimate_then_exploit': 'estimate then exploit', 'epsilon_greedy': 'epsilon-greedy', 'ucb': 'ucb'}
_N6_TASKS = {"my_estimate": "task A", "random_mirror": "task B", "estimate_then_exploit": "task C",
             "epsilon_greedy": "task D", "ucb": "task E"}
_N6_LABELS = {"random_mirror": "Random", "estimate_then_exploit": "estimate then exploit",
              "epsilon_greedy": "epsilon-greedy", "ucb": "UCB"}
_N6_DEFS = {"my_estimate": "def my_estimate(row):",
            "random_mirror": "def random_mirror(p, moods, next_number):",
            "estimate_then_exploit": "def estimate_then_exploit(p, moods, next_number, n):",
            "epsilon_greedy": "def epsilon_greedy(p, moods, next_number, eps):",
            "ucb": "def ucb(p, moods, next_number):"}
_N6_KNOWN = {"my_estimate": "row, taps, dot and len",
             "random_mirror": "p, moods, next_number, rows, morning, product, dot and tap",
             "estimate_then_exploit": "p, moods, next_number, n, rows, test, winner, morning, dot, tap, estimates "
                                      "and pick_highest",
             "epsilon_greedy": "p, moods, next_number, eps, rows, morning, spin, product, dot, tap, estimates and "
                               "pick_highest",
             "ucb": "p, moods, next_number, rows, morning, scores, row, bonus, product, dot, tap, estimate, sqrt and "
                    "pick_highest"}
_N6_KIND_HINTS = {
    "r": "each morning: product = int(next_number() * len(p)), one number from the hat, then tap and append for that "
         "product.",
    "t": "the test comes from round_robin, the given line at the top of the function: keep it as it was given.",
    "i": "the initialization comes from round_robin(p, moods, 1), the given line at the top: keep it as it was given.",
    "e": "the winner is pick_highest(estimates(rows), next_number), picked once after the test, before the loop; every "
         "remaining morning shows it: dot = tap(p[winner], moods[morning]).",
    "h": "when spin < eps, a card from the hat: product = int(next_number() * len(p)), a new number drawn after the "
         "spin.",
    "b": "when the spin is not below eps, the highest estimate: product = pick_highest(estimates(rows), next_number).",
    "u": "for each row, bonus = 1 / sqrt(len(row)) and its score is estimate(row) + bonus; then "
         "product = pick_highest(scores, next_number).",
}
_N6_GAPS = {}
_N6_LOG = []
_N6_TAP = tap


class _N6Prob(float):
    """A probability that knows its row, so the check sees which product is shown."""


class _N6Mood(float):
    """A mood that knows its morning."""


def _n6_tap(prob, mood):
    """The kit's tap, which also writes down the row and the morning of each answer."""
    if hasattr(prob, "morning") or hasattr(mood, "row"):
        _N6_LOG.append((getattr(mood, "row", None), getattr(prob, "morning", None), True))
    else:
        _N6_LOG.append((getattr(prob, "row", None), getattr(mood, "morning", None), False))
    return _N6_TAP(prob, mood)


def _n6_short(value):
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n6_ordinal(n):
    if n % 100 in (11, 12, 13):
        return "%dth" % n
    return "%d%s" % (n, {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th"))


def _n6_morning(t):
    return "morning %d (the %s of the season)" % (t, _n6_ordinal(t + 1))


def _n6_number(x):
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def _n6_product(test, k):
    return "the %s (row %d, p = %s)" % (NAMES[k].lower(), k, test["p"][k])


def _n6_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N6_GAPS:
        return _N6_GAPS[code]
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N6_GAPS[code] = gap
    return gap


def _n6_is_board(value, rows=None):
    if not isinstance(value, list) or len(value) == 0:
        return False
    if rows is not None and len(value) != rows:
        return False
    for row in value:
        if not isinstance(row, list):
            return False
        for x in row:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return False
    return True


def _n6_error_line(fn, error):
    import traceback
    try:
        line = None
        for frame in traceback.extract_tb(error.__traceback__):
            if frame.filename == fn.__code__.co_filename and frame.line:
                line = frame.line.strip()
        return line
    except Exception:
        return None


def _n6_run(fn, inputs, record=False):
    """Run fn on the inputs with its prints silenced, and with the recording tap when record is True. Gives back
    (answer, error, line, log). Never raises."""
    import contextlib, io
    names = getattr(fn, "__globals__", None)
    old = None
    if record and names is not None:
        old = names.get("tap")
        names["tap"] = _n6_tap
    del _N6_LOG[:]
    answer, error, line = None, None, None
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*inputs)
    except Exception as e:
        error = "%s: %s" % (type(e).__name__, e)
        line = _n6_error_line(fn, e)
    finally:
        if record and names is not None:
            names["tap"] = old
    return answer, error, line, list(_N6_LOG)


def _n6_inputs(name, test):
    """Fresh inputs for a test, and a counter of the numbers drawn from next_number."""
    p = []
    for k in range(len(test["p"])):
        x = _N6Prob(test["p"][k])
        x.row = k
        p.append(x)
    moods = []
    raw = season_moods(test["seed"])
    for t in range(len(raw)):
        m = _N6Mood(raw[t])
        m.morning = t
        moods.append(m)
    stream = strategy_numbers(test["seed"], _N6_STREAMS[name])
    drawn = [0]

    def next_number():
        drawn[0] = drawn[0] + 1
        return stream()
    inputs = [p, moods, next_number]
    if test["extra"] is not None:
        inputs.append(test["extra"])
    return inputs, drawn


def _n6_board(test, upto):
    """The game's board after the first upto mornings of a test."""
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for t in range(upto):
        rows[int(test["shown"][t])].append(int(test["dots"][t]))
    return rows


def _n6_board_text(test, t):
    """The board before morning t, in words: the dots and the estimate of every row."""
    parts = []
    rows = _n6_board(test, t)
    for k in range(len(rows)):
        if len(rows[k]) == 0:
            parts.append("%s no dots" % EMOJI[k])
        else:
            parts.append("%s %d %s, estimate %s" % (EMOJI[k], len(rows[k]), "dot" if len(rows[k]) == 1 else "dots",
                                                    _n6_number(sum(rows[k]) / len(rows[k]))))
    return "; ".join(parts)


def _n6_error_hint(name, error, line):
    text = (line or "").replace(" ", "")
    if "positional argument" in error:
        return "keep the first line of the function as it was given: " + _N6_DEFS[name]
    if error.startswith("NameError"):
        if "sqrt" in error:
            return "sqrt comes from the mirror's kit: run the SETUP cells at the top, then this cell and the check."
        return "Python does not know a name in your function: check its spelling. Inside it you can use %s." % (
            _N6_KNOWN[name])
    if error.startswith("ZeroDivisionError"):
        return "a row that was never shown has no showings: keep the given lines that return None for it."
    if error.startswith("TypeError"):
        if "indices must be integers" in error and "float" in error:
            if name == "random_mirror" or "next_number()*" in text:
                return "a row number is a whole number: int(next_number() * len(p)) turns the number into one."
            return ("pick_highest gives the row of the highest value, a whole number. max gives the value itself: "
                    "use pick_highest(..., next_number).")
        if "not callable" in error:
            return "next_number is a function: next_number() with brackets gives the next number."
        if "not supported between" in error:
            return ("tap compares two numbers: the probability of the product shown, p[product], and the mood of "
                    "this morning, moods[morning].")
        if "len()" in error:
            return "len works on a list: len(row) is the showings of a row."
    if error.startswith("IndexError"):
        return ("an index went past the end of a list. Python counts from 0: the rows are 0 to %d, the mornings 0 to "
                "%d." % (len(NAMES) - 1, SEASON - 1))
    if error.startswith("AttributeError") and "append" in error:
        return "append works on a list: rows[product] is the row of the product shown, and rows[product].append(dot)."
    return "compare your code with the steps above the cell, then open the hints."


def _n6_shape(answer, test):
    """(what, hint) when answer is not a board of the right size, or None."""
    rows = len(test["p"])
    if answer is None:
        return ("your function gave no answer (None)", "end the function with return rows, after the loop over the "
                                                         "mornings.")
    if not isinstance(answer, list) or len(answer) == 0:
        return ("your function gave %s, and a board is a list of rows" % _n6_short(answer),
                "return rows, the board: one list per product.")
    for row in answer:
        if not isinstance(row, list):
            return ("your function gave %s, a board with numbers next to its rows" % _n6_short(answer),
                    "add each dot to the row of the product shown: rows[product].append(dot).")
    if len(answer) != rows:
        return ("your function gave a board of %d rows for %d products" % (len(answer), rows),
                "one row per product: keep the given start of the board.")
    for r in range(rows):
        for x in answer[r]:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return ("your function gave a board whose row %d holds %s" % (r, _n6_short(x)),
                        "each dot is her answer, 1 or 0: dot = tap(p[product], moods[morning]).")
    return None


def _n6_mornings(name, test, log, slip):
    """(what, hint) from the answers that the recording tap wrote down, or None when they match the game's. slip:
    the key of the usual slip that the mirror plays, or None."""
    mornings = len(test["shown"])
    for row, morning, swapped in log:
        if swapped:
            return ("your mirror called tap with the mood first",
                    "tap takes the probability first, then the mood: dot = tap(p[product], moods[morning]).")
    for t in range(mornings):
        if t >= len(log):
            return ("your mirror gave %d answers in %d mornings" % (len(log), mornings),
                    "one answer per morning, up to the last morning: call tap once in each round of the loop.")
        row, morning, swapped = log[t]
        if morning is None:
            return ("on %s your mirror used a mood that is not from moods" % _n6_morning(t),
                    "use moods, the moods the function gets: dot = tap(p[product], moods[morning]).")
        if morning != t:
            return ("its answer number %d read the mood of morning %d, and that answer belongs to morning %d"
                    % (t + 1, morning, t),
                    "each morning has its own mood: dot = tap(p[product], moods[morning]).")
        if row is None:
            return ("on %s your mirror used a probability that is not from p" % _n6_morning(t),
                    "use p, the probabilities the function gets: the check also tries %s." % "Rapunzel")
        right = int(test["shown"][t])
        if row != right:
            kind = test["kinds"][t]
            what = "on %s your mirror showed %s, and the game's %s showed %s" % (
                _n6_morning(t), _n6_product(test, row), _N6_LABELS[name], _n6_product(test, right))
            if kind in "hb":
                spin = test["notes"][t]
                what = what + ": the spin was %s, %s eps = %s" % (spin, "below" if kind == "h" else "not below",
                                                                  test["extra"])
            if kind == "u" or (kind == "e" and t > 0 and test["kinds"][t - 1] == "t"):
                what = what + ". The board before that morning: " + _n6_board_text(test, t)
            hint = _N6_KIND_HINTS[kind]
            if slip is not None:
                hint = _N6_SLIPS[name][slip]
            return what, hint
    if len(log) > mornings:
        return ("your mirror gave %d answers in %d mornings" % (len(log), mornings),
                "one answer per morning: call tap once in each round of the loop.")
    return None


def _n6_played(log):
    """The rows shown, morning by morning, as one text: "01234..." ("x" for a probability not from p)."""
    played = ""
    for row, morning, swapped in log:
        played = played + ("x" if row is None else str(row))
    return played


def _n6_slip(name, fn):
    """The key of the usual slip whose seasons fn plays on every test, or None."""
    logs = []
    for test in _N6_TESTS[name]:
        inputs, drawn = _n6_inputs(name, test)
        answer, error, line, log = _n6_run(fn, inputs, True)
        if error is not None:
            return None
        logs.append(_n6_played(log))
    for key in _N6_SLIPS[name]:
        same = True
        for k in range(len(logs)):
            test = _N6_TESTS[name][k]
            if logs[k] != test["slips"].get(key, test["shown"]):
                same = False
        if same:
            return key
    return None


def _n6_rows_differ(answer, test):
    expected = _n6_board(test, len(test["shown"]))
    for k in range(len(expected)):
        if answer[k] != expected[k]:
            return ("the products and her answers are right, and the row of %s has %d dots where the game's has %d"
                    % (_n6_product(test, k), len(answer[k]), len(expected[k])),
                    "add each dot to the row of the product shown, once: rows[product].append(dot).")
    return ("your function gave %s" % _n6_short(answer), "compare your code with the steps above the cell.")


def _n6_check_mirror(name, fn):
    for test in _N6_TESTS[name]:
        inputs, drawn = _n6_inputs(name, test)
        answer, error, line, log = _n6_run(fn, inputs, True)
        expected = _n6_board(test, len(test["shown"]))
        if error is None and _n6_is_board(answer) and answer == expected:
            continue
        if _n6_has_gap(fn) or answer is Ellipsis or "ellipsis" in str(error).lower():
            print("⏳ %s is not finished yet: replace each ... with your code, run its cell, then run this check "
                  "again." % name)
            return
        head = "%s, %s" % (test["call"], test["about"])
        if error is not None:
            where = ""
            if line:
                where = " on the line  %s" % line
            print("❌ %s: your function stopped with an error%s. %s" % (head, where, error))
            print("   💡 Hint: " + _n6_error_hint(name, error, line))
            return
        problem = _n6_shape(answer, test)
        slip = None
        if problem is None:
            slip = _n6_slip(name, fn)
        if slip is not None:
            problem = _n6_mornings(name, test, log, slip)
        if problem is None and drawn[0] == 0 and test["draws"] > 0:
            names = getattr(getattr(fn, "__code__", None), "co_names", ())
            if "random" in names or "randint" in names or "choice" in names:
                hint = ("use next_number, the stream the function gets. Python's own random numbers give another "
                        "season than the game's.")
            elif name == "random_mirror":
                hint = "draw each morning's card from the hat: product = int(next_number() * len(p))."
            else:
                hint = ("on a tie the game's mirror draws a number to choose among the tied rows: pick_highest(..., "
                        "next_number) does that.")
            problem = ("your mirror never drew a number from next_number", hint)
        if problem is None and name == "random_mirror" and drawn[0] != len(test["shown"]):
            problem = ("your mirror drew %d numbers in %d mornings" % (drawn[0], len(test["shown"])),
                       "call next_number() exactly once each morning, for the card.")
        if problem is None:
            problem = _n6_mornings(name, test, log, None)
        if problem is None:
            problem = _n6_rows_differ(answer, test)
        what, hint = problem
        print("❌ %s: %s." % (head, what))
        print("   💡 Hint: " + hint)
        return
    print("✅ " + _N6_SUCCESS[name])


def _n6_check_estimate(fn):
    for row, expected in _N6_EST_TESTS:
        answer, error, line, log = _n6_run(fn, [list(row)])
        same = False
        if error is None:
            if expected is None:
                same = answer is None
            else:
                same = isinstance(answer, (int, float)) and not isinstance(answer, bool) and abs(answer - expected) < 1e-9
        if same:
            continue
        if _n6_has_gap(fn) or answer is Ellipsis or "ellipsis" in str(error).lower():
            print("⏳ my_estimate is not finished yet: replace each ... with your code, run its cell, then run this "
                  "check again.")
            return
        head = "my_estimate(%s)" % _n6_short(row)
        if error is not None:
            where = ""
            if line:
                where = " on the line  %s" % line
            print("❌ %s: your function stopped with an error%s. %s" % (head, where, error))
            print("   💡 Hint: " + _n6_error_hint("my_estimate", error, line))
            return
        if expected is None:
            print("❌ %s gave %s: a row that was never shown has no estimate yet." % (head, _n6_short(answer)))
            print("   💡 Hint: keep the given lines at the top: if len(row) == 0, return None.")
            return
        taps = sum(row)
        print("❌ %s gave %s. The estimate is %s: %d taps / %d showings." % (
            head, _n6_short(answer), _n6_number(expected), taps, len(row)))
        if answer is None:
            hint = "end the function with return and the estimate, taps / len(row)."
        elif len(row) > 1 and answer != taps and (answer == row[0] / len(row) or answer == row[0]):
            hint = ("your function stopped after the first dot. Put return after the loop, at the left edge of the "
                    "function's body, under for.")
        elif len(row) > 1 and answer == row[0] / len(row) and row[0] == 0 and taps > 0:
            hint = ("your function stopped after the first dot. Put return after the loop, at the left edge of the "
                    "function's body, under for.")
        elif answer == taps and len(row) > 1:
            hint = "these are the taps. Divide them by the showings, len(row): return taps / len(row)."
        elif answer == len(row) and taps != len(row):
            hint = "these are the showings. The taps are the 1s of the row: taps = taps + dot adds each of them."
        elif len(row) > 1 and answer == row[-1] / len(row):
            hint = "add every dot to the taps: taps = taps + dot. taps = dot keeps only the last one."
        else:
            hint = "the estimate is taps / showings: taps = taps + dot in the loop, then return taps / len(row)."
        print("   💡 Hint: " + hint)
        return
    print("✅ " + _N6_SUCCESS["my_estimate"])


def _n6_check(name, fn):
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    try:
        if name == "my_estimate":
            _n6_check_estimate(fn)
        else:
            _n6_check_mirror(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two SETUP cells at the top again, then this cell." % error)


def check_my_estimate(fn):
    """Task A: compares my_estimate with the game's estimate on the rows of section 3's board and a few more."""
    _n6_check("my_estimate", fn)


def check_random_mirror(fn):
    """Task B: plays random_mirror on four seasons, both customers, and compares them with the game's."""
    _n6_check("random_mirror", fn)


def check_estimate_then_exploit(fn):
    """Task C: plays estimate_then_exploit on four seasons, both customers and three values of n."""
    _n6_check("estimate_then_exploit", fn)


def check_epsilon_greedy(fn):
    """Task D: plays epsilon_greedy on four seasons, both customers and two values of eps."""
    _n6_check("epsilon_greedy", fn)


def check_ucb(fn):
    """Task E: plays ucb on four seasons, both customers."""
    _n6_check("ucb", fn)


def play(mirror, *inputs):
    """Runs your function on the inputs and gives its answer: play(random_mirror, cinderella, moods, next_number) is
    random_mirror(cinderella, moods, next_number). When the function is not finished yet, stops with an error or
    gives something that is not a board, play prints one calm line and gives None, so the cells below still run."""
    try:
        if not callable(mirror):
            print("⏳ play needs your function itself, as in play(random_mirror, cinderella, moods, next_number).")
            return None
        name = getattr(mirror, "__name__", "your mirror")
        if _n6_has_gap(mirror):
            task = ""
            if name in _N6_TASKS:
                task = " (%s)" % _N6_TASKS[name]
            print("⏳ %s is not finished yet%s, so there is nothing to show. Finish it, run its cell, then run this "
                  "cell again." % (name, task))
            return None
        copies = []
        for x in inputs:
            if isinstance(x, list):
                copies.append(list(x))
            else:
                copies.append(x)
        answer, error, line, log = _n6_run(mirror, copies)
        if error is not None:
            print("❌ %s stopped with an error, %s. Its check above says more." % (name, error))
            return None
        if name == "my_estimate":
            return answer
        rows = None
        if len(copies) > 0 and isinstance(copies[0], list):
            rows = len(copies[0])
        if not _n6_is_board(answer, rows):
            print("❌ %s gave %s, which is not a board. Its check above says more." % (name, _n6_short(answer)))
            return None
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the two SETUP cells at the top again, then this cell." % error)
        return None


print("✅ The checks of notebook 6 are ready.")

✅ The checks of notebook 6 are ready.


## The customers and the season

The game's two customers, with the probability that each of them taps each product, in board order: row 0 is the
robot vacuum and row 4 the satnav, since Python counts from 0. The numbers are invented for the game.

A season is 100 mornings. Each morning the mirror shows one product, and the customer has a mood, a number between
0 and 1: she taps when her mood is below the product's probability. `season_moods(RACE)` gives her moods on the
game's race season, the season of every comparison in the game, and `tap(p, mood)` gives her answer, 1 for a tap
and 0 for no tap.

A **board** is one list per product, in board order: a product's row holds a 1 for each tap and a 0 for each
showing without a tap, in the order of the showings. It is the board of the game, written in Python.

In [3]:
# The probability that each customer taps each product, in board order (invented for the game).
# Only the Queen's mirror may read them: every other mirror sees only the dots of its board.
cinderella = [0.6, 0.2, 0.2, 0.4, 0.2]
rapunzel = [0.4, 0.8, 0.2, 0.2, 0.2]

for i in range(len(NAMES)):
    print("row", i, "   Cinderella", cinderella[i], "   Rapunzel", rapunzel[i], "   ", EMOJI[i], NAMES[i])

# The race season of the game: her mood on each of its mornings
moods = season_moods(RACE)
print()
print("Mornings in the race season:", len(moods))
print("Her mood on the first morning:", moods[0])
print("Shown the robot vacuum, does she tap?", tap(cinderella[0], moods[0]))

row 0    Cinderella 0.6    Rapunzel 0.4     🤖 Robot vacuum
row 1    Cinderella 0.2    Rapunzel 0.8     🪜 Rope ladder
row 2    Cinderella 0.2    Rapunzel 0.2     🍃 Leaf blower
row 3    Cinderella 0.4    Rapunzel 0.2     🛏️ Memory-foam mattress
row 4    Cinderella 0.2    Rapunzel 0.2     🧭 Satnav

Mornings in the race season: 100
Her mood on the first morning: 0.7226773640140891
Shown the robot vacuum, does she tap? 0


## 🎯 Task A. The estimate

The estimate of a product is the share of its showings that got a tap: taps / showings. The row `[0, 1, 0, 1, 1]` has 3 taps
in 5 showings, an estimate of 0.6. A row that was never shown has no estimate yet, and the given lines give
`None` for it.

The kit already has `estimate(row)`, which tasks C to E use. Here you write your own, `my_estimate(row)`, and the
check compares it with the game's. Two gaps: add each dot to the taps, then return the estimate.

**💡 Hint 1 for task A**

- `taps` starts at 0. Each `dot` is 1 (a tap) or 0 (no tap), so adding it to `taps` counts the taps.
- The showings are the dots of the row: `len(row)`. The estimate is the taps divided by the showings.

**💡 Hint 2 for task A: fill in the blanks (if you are still stuck)**

- By hand on the row `[1, 0, 1]`: `taps` goes 0, then 1, then 1, then 2. The row has 3 dots, so its estimate is 2 / 3.
- Now fill in the blanks `___`:

```python
taps = taps + ___      # what does one dot add to the count?
return ___ / ___       # the taps divided by the showings
```

In [ ]:
# 🎯 YOUR TURN, task A: the estimate of a row
def my_estimate(row):
    # row: one product's dots, 1 for a tap and 0 for no tap, for example [0, 1, 0, 1, 1]
    # Returns its estimate, taps / showings, or None for a row that was never shown.
    if len(row) == 0:
        return None             # never shown: no estimate yet

    taps = 0
    for dot in row:
        taps = ...              # 🎯 YOUR TURN: add this dot to the taps
    return ...                  # 🎯 YOUR TURN: the estimate, taps / showings (the showings are len(row))

In [ ]:
check_my_estimate(my_estimate)

## The board after the test

In section 3 of the game, estimate then exploit tested every product 5 times on the race season at Cinderella's. The
cell shows that board; the number on the left of each row is the kit's estimate.

💭 **Think first.** The robot vacuum's row is `[0, 1, 0, 1, 1]` and the memory-foam mattress's `[1, 0, 1, 0, 0]`. What are their estimates?

In [ ]:
# The game's board after the test of section 3 (estimate then exploit, n = 5), race season, Cinderella's
test_board = [
    [0, 1, 0, 1, 1],     # 🤖 Robot vacuum
    [0, 0, 0, 0, 0],     # 🪜 Rope ladder
    [0, 0, 0, 0, 0],     # 🍃 Leaf blower
    [1, 0, 1, 0, 0],     # 🛏️ Memory-foam mattress
    [0, 0, 0, 0, 0],     # 🧭 Satnav
]
show_board(test_board)

vacuum = play(my_estimate, test_board[0])
if vacuum != None:
    print("Your estimate of the robot vacuum:", vacuum)
    print("Your estimate of the memory-foam mattress:", my_estimate(test_board[3]))

**Reading the result.** The robot vacuum: 3 taps in 5 showings, 0.6. The memory-foam mattress: 2 in 5, 0.4. The other three got no tap,
so their estimates are 0. Your function agrees with the left column of the board, and the robot vacuum leads.

## 🎯 Task B. Random

Every mirror plays a season the same way. Each morning it chooses a product; her answer is
`dot = tap(p[product], moods[morning])`; and the dot goes at the end of that product's row,
`rows[product].append(dot)`. The cell below gives you the Queen's mirror, which reads `p` and shows the product
with the highest probability every morning. Run it, and use it as the model for your mirrors.

Random draws a card from a hat every morning, so each product has the same chance, 1 in 5. In Python the hat is
`next_number`, a stream of random numbers between 0 and 1: each call `next_number()` gives the next number, and
`int(next_number() * len(p))` turns it into a row from 0 to 4 (`int` drops the decimals).
`strategy_numbers(RACE, "random")` is Random's own hat in the race season, as in the game.

In [7]:
# Given: the Queen's mirror. It reads p and shows the product with the highest probability every morning.
def queen_mirror(p, moods):
    best = 0
    for i in range(len(p)):
        if p[i] > p[best]:
            best = i                            # the row with the highest probability so far
    rows = []
    for i in range(len(p)):
        rows.append([])                         # the board: one empty row per product
    for morning in range(len(moods)):
        dot = tap(p[best], moods[morning])      # her answer: 1 for a tap, 0 for no tap
        rows[best].append(dot)                  # one more dot in the best product's row
    return rows


# Given: all the taps of a board, added up
def count_taps(rows):
    taps = 0
    for row in rows:
        for dot in row:
            taps = taps + dot
    return taps

**💡 Hint 1 for task B**

- `next_number()` gives a number between 0 and 1. Times `len(p)` it is between 0 and 5, and `int(...)` drops the decimals: a row from 0 to 4.
- Call `next_number()` once each morning, inside the loop.
- Her answer and the update work as in `queen_mirror` above, with `product` in the place of `best`.

**💡 Hint 2 for task B: fill in the blanks (if you are still stuck)**

- The last two lines of the loop in `queen_mirror` already do her answer and the update. Yours are the same, with one name changed.
- Fill in the blanks `___`:

```python
product = int(___ * len(p))         # one new number from the hat
dot = tap(p[___], moods[morning])   # which row was shown this morning?
rows[___].append(dot)
```

In [ ]:
# 🎯 YOUR TURN, task B: Random
def random_mirror(p, moods, next_number):
    # p: the probability of a tap for each product. Only tap reads it: the card never looks at it.
    # moods: her mood on each morning. next_number: the hat, next_number() gives the next number.
    # Returns the board of the whole season.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product

    for morning in range(len(moods)):
        product = ...                           # 🎯 YOUR TURN: a card from the hat, a row from 0 to 4
        dot = ...                               # 🎯 YOUR TURN: her answer to that product this morning
        ...                                     # 🎯 YOUR TURN: add dot at the end of that product's row
    return rows

In [ ]:
check_random_mirror(random_mirror)

💭 **Think first.** Random shows each of the five products on about one morning in five, whatever she likes. About
what rate do you expect from Random? And from the Queen's mirror, which shows the robot vacuum (p = 0.6) every morning?

In [ ]:
# The race season at Cinderella's: the Queen's mirror and Random meet the same moods
moods = season_moods(RACE)
queen_rows = play(queen_mirror, cinderella, moods)
random_rows = play(random_mirror, cinderella, moods, strategy_numbers(RACE, "random"))
if random_rows != None:
    print("The Queen's mirror:", count_taps(queen_rows), "taps, a rate of", rate(queen_rows))
    print("Random:            ", count_taps(random_rows), "taps, a rate of", rate(random_rows))

**Reading the result.** The Queen's mirror: 60 taps, a rate of 0.6. Random: 31 taps, 0.31, close to the average of
her probabilities, (0.6 + 0.2 + 0.2 + 0.4 + 0.2) / 5 = 0.32. Random spread the mornings over all the products (30, 15, 14, 21 and 20 showings) and never
learns from its board. These are the game's numbers.

**💭 Why the same moods?**

- Every mirror of a season meets the same moods, as in the game.
- A morning with a low mood is a lucky morning for every mirror: she taps any product whose probability is above her mood.
- So a gap between two rates comes from the mirrors' choices, with less luck mixed in.

## 🎯 Task C. Estimate then exploit

Estimate then exploit first **tests**: n rounds, and in each round every product once, in board order. The first
round is the **initialization**. Then it picks the product with the highest estimate, once, and **exploits** it:
it shows that product on every remaining morning.

The cell below gives the test, `round_robin(p, moods, rounds)`. `morning % len(p)` is the remainder of morning
divided by 5, so the products come in turn: 0, 1, 2, 3, 4, then 0 again. The cell shows the board after the
initialization on the race season.

The kit gives two helpers for the pick. `estimates(rows)` is the list of the estimates of every row, and
`pick_highest(values, next_number)` gives the row with the highest value. When rows tie, it picks one of them at
random with a number from `next_number`, exactly as the game does. `strategy_numbers(RACE, "estimate then
exploit")` is the mirror's own stream of numbers in the race season.

In [11]:
# Given: the test. It plays `rounds` rounds; in each round every product is shown once, from the top down.
# round_robin(p, moods, 1) is the initialization: each product once.
def round_robin(p, moods, rounds):
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product
    for morning in range(rounds * len(p)):
        product = morning % len(p)              # in turn: 0, 1, 2, 3, 4, then 0 again
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row
    return rows


# The initialization on the race season at Cinderella's
show_board(round_robin(cinderella, moods, 1))

    p̂
    0  🤖  ⚪
    0  🪜  ⚪
    0  🍃  ⚪
    1  🛏️  🟠
    0  🧭  ⚪


**💡 Hint 1 for task C**

- `estimates(rows)` is a list with the estimate of every row.
- `pick_highest(estimates(rows), next_number)` gives the row of the highest estimate. Pick once, before the loop.
- Each remaining morning: her answer to `winner` with `tap`, then `append`, as in `queen_mirror` with `winner` in the place of `best`.

**💡 Hint 2 for task C: fill in the blanks (if you are still stuck)**

- `estimates(rows)` gives a list such as `[0.6, 0.0, 0.0, 0.4, 0.0]`; `pick_highest` turns a list like that into a row number.
- Fill in the blanks `___`:

```python
winner = pick_highest(___, next_number)   # the list of the estimates of every row
dot = tap(p[___], moods[morning])          # every remaining morning shows the same row
rows[___].append(___)
```

In [ ]:
# 🎯 YOUR TURN, task C: estimate then exploit
def estimate_then_exploit(p, moods, next_number, n):
    # p: the probabilities, only tap reads them. moods: her mood on each morning.
    # next_number: the mirror's own stream of numbers, for a tie. n: the showings of each product in the test.
    # Returns the board of the whole season.

    # 1. Initialize and test: n rounds, every product once per round (given)
    rows = round_robin(p, moods, n)
    test = n * len(p)                           # the mornings of the test

    # 2. The winner: the row with the highest estimate, picked once
    winner = ...                                # 🎯 YOUR TURN: use estimates(rows) and pick_highest

    # 3. Exploit: the winner on every remaining morning
    for morning in range(test, len(moods)):
        dot = ...                               # 🎯 YOUR TURN: her answer to the winner this morning
        ...                                     # 🎯 YOUR TURN: add dot at the end of the winner's row
    return rows

In [ ]:
check_estimate_then_exploit(estimate_then_exploit)

💭 **Think first.** With n = 5 the test takes 25 mornings, 5 for each product. After it, the robot vacuum leads with 0.6
and the memory-foam mattress follows with 0.4 (task A). Which product wins, and about how many taps do you expect from the 75
remaining mornings?

In [ ]:
# Estimate then exploit on the race season at Cinderella's, with the game's n
n = 5
ete_rows = play(estimate_then_exploit, cinderella, moods, strategy_numbers(RACE, "estimate then exploit"), n)
if ete_rows != None:
    print("Estimate then exploit, n =", n, ":", count_taps(ete_rows), "taps, a rate of", rate(ete_rows))

**Reading the result.** The test brought 5 taps in 25 mornings. Then the robot vacuum won and got 41 taps in the 75
remaining mornings. In all 46 taps, a rate of 0.46, between Random's 0.31 and the Queen's mirror's 0.6: the game's
numbers. The test is the price of not knowing p: 20 of its 25 mornings went to products she taps less often.

## 🎯 Task D. Epsilon-greedy

Epsilon-greedy initializes, and then spins a roulette every morning: `spin = next_number()`. When the spin is
below `eps` (the small slice, the hat 🎩), it takes a card from the hat, any product at random, as Random does.
Otherwise (the large slice) it shows the product with the highest estimate, as estimate then exploit does after
its test. The card from the hat is a new number, drawn after the spin. In the game eps is 0.2.

**💡 Hint 1 for task D**

- The small slice: the spin is below eps, `spin < eps`.
- The card from the hat: `int(next_number() * len(p))`, as in task B.
- The highest estimate: `pick_highest(estimates(rows), next_number)`, as in task C.

**💡 Hint 2 for task D: fill in the blanks (if you are still stuck)**

- Each branch is a line you already wrote: the `else` branch is the pick of task C, the other one the card of task B.
- Fill in the blanks `___`:

```python
if spin ___ eps:        # < or > ? The small slice is the numbers below eps
    product = ___       # the card from the hat (task B)
else:
    product = ___       # the highest estimate (task C)
```

In [ ]:
# 🎯 YOUR TURN, task D: epsilon-greedy
def epsilon_greedy(p, moods, next_number, eps):
    # p, moods and next_number as in task C. eps: the size of the small slice, the hat.
    # Returns the board of the whole season.
    rows = round_robin(p, moods, 1)             # initialize: each product once (given)

    for morning in range(len(p), len(moods)):
        spin = next_number()                    # where the needle stops: a number between 0 and 1
        if ...:                                 # 🎯 YOUR TURN: the needle stops on the small slice
            product = ...                       # 🎯 YOUR TURN: a card from the hat, as in task B
        else:
            product = ...                       # 🎯 YOUR TURN: the highest estimate, as in task C
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # update: the next spin sees the new estimate
    return rows

In [ ]:
check_epsilon_greedy(epsilon_greedy)

💭 **Think first.** After the initialization only the memory-foam mattress has a tap, so the large slice shows the memory-foam mattress. The mirror
spins 95 times, and about one spin in 5 lands on the hat. How can it ever find the robot vacuum, Cinderella's favourite?

In [ ]:
# Epsilon-greedy on the race season at Cinderella's, with the game's eps
eps = 0.2
eg_rows = play(epsilon_greedy, cinderella, moods, strategy_numbers(RACE, "epsilon-greedy"), eps)
if eg_rows != None:
    print("Epsilon-greedy, eps =", eps, ":", count_taps(eg_rows), "taps, a rate of", rate(eg_rows))

**Reading the result.** 47 taps, a rate of 0.47, the game's number. The hat came up 16 times in 95 spins. The
large slice showed the mattress morning after morning, until a card from the hat showed the robot vacuum on morning 17:
she tapped, the robot vacuum took the lead, and the large slice showed it from then on. The robot vacuum got 71 of the 100 mornings.

## 🎯 Task E. UCB

UCB gives every product a **score** each morning: its estimate plus an exploration bonus, 1 / square root of its
showings. The fewer dots in a row, the bigger its bonus: a row with 1 dot gets 1, a row with 4 dots 1 / 2. The
mirror initializes, and then each morning it shows the product with the highest score, a tie broken at random
with `pick_highest`. The kit gives the square root, `sqrt`: `sqrt(9)` is 3.0. `estimate(row)` from the kit, or your
`my_estimate(row)`, gives the estimate of a row.

**💡 Hint 1 for task E**

- The showings of a row are `len(row)`, so its bonus is `1 / sqrt(len(row))`.
- The score of a row is `estimate(row) + bonus`; add it to the list with `scores.append(...)`.
- `pick_highest(scores, next_number)` gives the row of the highest score.

**💡 Hint 2 for task E: fill in the blanks (if you are still stuck)**

- By hand on the row `[1, 0, 0, 1]`: 4 showings, estimate 2 / 4 = 0.5, bonus 1 / sqrt(4) = 0.5, score 0.5 + 0.5 = 1.0.
- Fill in the blanks `___`:

```python
bonus = 1 / sqrt(___)              # the number of showings of this row
scores.append(___ + bonus)         # the estimate of this row
product = pick_highest(___, next_number)
```

In [ ]:
# 🎯 YOUR TURN, task E: UCB
def ucb(p, moods, next_number):
    # p, moods and next_number as in task C.
    # Returns the board of the whole season.
    rows = round_robin(p, moods, 1)             # initialize: each product once (given)

    for morning in range(len(p), len(moods)):
        scores = []
        for row in rows:
            bonus = ...                         # 🎯 YOUR TURN: 1 / square root of the showings of this row
            scores.append(...)                  # 🎯 YOUR TURN: its score, the estimate of the row + bonus
        product = ...                           # 🎯 YOUR TURN: the row with the highest score (pick_highest)
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # update: its estimate and its bonus change
    return rows

In [ ]:
check_ucb(ucb)

💭 **Think first.** After the initialization the memory-foam mattress scores 1 + 1 = 2 and every other product 0 + 1 = 1. Which
product does UCB show on morning 5? A product left alone keeps its bonus, while the bonus of the product shown
shrinks. How many of the 100 mornings do you expect the robot vacuum to get?

In [ ]:
# UCB on the race season at Cinderella's
ucb_rows = play(ucb, cinderella, moods, strategy_numbers(RACE, "ucb"))
if ucb_rows != None:
    print("UCB:", count_taps(ucb_rows), "taps, a rate of", rate(ucb_rows))

**Reading the result.** 52 taps, a rate of 0.52, the game's number. UCB first showed the memory-foam mattress, the highest score; as
its estimate fell and the bonus of the others stayed, every product got its turn, and the robot vacuum took over. In the
end the robot vacuum got 78 mornings and the other products 3, 3, 13 and 3.

## The race

Every mirror on the race season at Cinderella's, all on the same moods, with the game's n = 5 and eps = 0.2: the same
race as the game's last scene. The cell shows the board of each mirror and its rate.

💭 **Think first.** Rank the five mirrors before you run the cell.

In [ ]:
# The race: every mirror on the race season at Cinderella's, all on the same moods
n = 5
eps = 0.2
moods = season_moods(RACE)
mirror_names = ["The Queen's mirror",
                "Random",
                "Estimate then exploit, n = 5",
                "Epsilon-greedy, eps = 0.2",
                "UCB"]

boards = []
boards.append(play(queen_mirror, cinderella, moods))
boards.append(play(random_mirror, cinderella, moods, strategy_numbers(RACE, "random")))
boards.append(play(estimate_then_exploit, cinderella, moods, strategy_numbers(RACE, "estimate then exploit"), n))
boards.append(play(epsilon_greedy, cinderella, moods, strategy_numbers(RACE, "epsilon-greedy"), eps))
boards.append(play(ucb, cinderella, moods, strategy_numbers(RACE, "ucb")))

for i in range(len(boards)):
    if boards[i] != None:
        print()
        print(mirror_names[i], ":", count_taps(boards[i]), "taps, a rate of", rate(boards[i]))
        show_board(boards[i])

**Reading the result.** The Queen's mirror 60, UCB 52, epsilon-greedy 47, estimate then exploit 46, Random 31: the
game's numbers, so a mirror of yours that gets them plays exactly as the game's. The boards tell the story:
Random spreads its dots over every row, estimate then exploit has 5 dots in each row and then one long row, and
epsilon-greedy and UCB give most mornings to the robot vacuum and a few to every other row. Only the Queen's mirror knows p; the other
four learn it from their boards and pay for the learning with mornings on other products.

## Many seasons

One season is one roll of luck: on another season the order of the learning mirrors can change.
`average_rates(p, first_seed, seasons)` plays every mirror on many seasons, one season per seed, and prints the
rate of each over all those mornings. It takes a few seconds.

💭 **Think first.** Will the order of the race season hold over 500 seasons?

In [22]:
# Given: every mirror over many seasons, one season per seed
def average_rates(p, first_seed, seasons):
    # Plays every mirror on each season and prints the rate of all those mornings, in the order of mirror_names.
    # In each season every mirror meets the same moods, and each has its own stream of numbers.
    taps = [0, 0, 0, 0, 0]
    for season in range(seasons):
        seed = first_seed + season
        moods = season_moods(seed)
        boards = []
        boards.append(play(queen_mirror, p, moods))
        boards.append(play(random_mirror, p, moods, strategy_numbers(seed, "random")))
        boards.append(play(estimate_then_exploit, p, moods, strategy_numbers(seed, "estimate then exploit"), n))
        boards.append(play(epsilon_greedy, p, moods, strategy_numbers(seed, "epsilon-greedy"), eps))
        boards.append(play(ucb, p, moods, strategy_numbers(seed, "ucb")))
        for i in range(len(boards)):
            if boards[i] == None:
                return                          # a mirror is not ready yet: play has said why
            taps[i] = taps[i] + count_taps(boards[i])
    mornings = seasons * SEASON
    print("Seasons:", seasons, "   mornings in all:", mornings)
    for i in range(len(taps)):
        print(mirror_names[i], ":", taps[i] / mornings)

In [ ]:
# 500 seasons at Cinderella's: seeds 1 to 500
average_rates(cinderella, 1, 500)

**Reading the result.** At Cinderella's: the Queen's mirror about 0.6, UCB 0.5, epsilon-greedy 0.47, estimate then exploit 0.45,
Random 0.32. The order of the race season holds on average, and UCB comes closest to the Queen's mirror.

💭 **Think first.** Rapunzel's favourite, the rope ladder at 0.8, stands far above her other products. Guess the rates before you
run the cell.

In [ ]:
# The same 500 seasons at Rapunzel's
average_rates(rapunzel, 1, 500)

**Reading the result.** At Rapunzel's: the Queen's mirror about 0.8, UCB 0.73, epsilon-greedy 0.66, estimate then exploit 0.64,
Random 0.36. The same order again. Random falls furthest behind here: it spends four mornings in five on products
she taps far less often than the ladder.

## Back to the game

You have written every mirror of the game in a few lines: the estimate, Random, estimate then exploit,
epsilon-greedy and UCB. Change `n` or `eps` in the cells above and run the race again to see how the learning
mirrors move.